# Extended Case 1 — Springville Herald

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: case-import
source("lessons/_includes/course-tools.R")
calls <- read.csv("data/business/extended-herald-calls.csv")
marketing <- read.csv("data/business/extended-herald-marketing.csv")
subscriptions <- read.csv("data/business/extended-herald-subscriptions.csv")
stopifnot(nrow(calls) == 90, nrow(marketing) == 24, nrow(subscriptions) == 24)
knitr::kable(data.frame(file = c("SH2", "SH13", "SH16"),
  observations = c(nrow(calls), nrow(marketing), nrow(subscriptions)),
  unit = c("Day", "Month: marketing", "Month: subscription base")))


- CSVs: `data/business/extended-herald-calls.csv`, `data/business/extended-herald-marketing.csv`, `data/business/extended-herald-subscriptions.csv`.
- SH2 chart-helper columns were excluded; all 90 daily call counts remain.
- `source()` loads the same R packages and plot style used in the core course.

## A. Describe daily calling activity


In [ ]:
call_summary <- calls %>% summarise(days = n(), mean_calls = mean(calls),
  median_calls = median(calls), sd_calls = sd(calls),
  min_calls = min(calls), max_calls = max(calls))
knitr::kable(call_summary, digits = 2)
ggplot(calls, aes(calls)) +
  geom_histogram(binwidth = 10, boundary = 0, fill = petra_ink, colour = "white") +
  labs(x = "Telephone calls per day", y = "Days")


- `summarise()` reports centre and spread across days.
- The standard deviation describes day-to-day variation; it is not a standard error.
- A histogram groups call counts; its vertical axis counts days.
- **Discuss:** Would the mean alone give the circulation manager enough information?

### Independent task A

1.  Calculate the first and third quartiles and the IQR of daily calls.
2.  Draw a boxplot and investigate any observations flagged by the 1.5-IQR rule; retain valid observations.
3.  Explain how changing the histogram bin width affects its appearance.
4.  Write three bullets about typical workload, variability and a limitation.

## B. Relate telemarketing hours to new subscriptions


In [ ]:
knitr::kable(data.frame(variable = c("Hours per month", "New subscriptions per month"),
  mean = sapply(marketing, mean), sd = sapply(marketing, sd),
  min = sapply(marketing, min), max = sapply(marketing, max)),
  row.names = FALSE, digits = 2)
ggplot(marketing, aes(hours, new_subscriptions)) +
  geom_point(colour = petra_ink) + geom_smooth(method = "lm", formula = y ~ x, colour = petra_ink) +
  labs(x = "Telemarketing hours per month", y = "New subscriptions per month")
model <- lm(new_subscriptions ~ hours, data = marketing)
summary(model)
confint(model)


- `lm(y ~ x)` fits a straight-line relationship.
- The slope is the estimated difference in new monthly subscriptions per additional telemarketing hour.
- Observational association does not establish that increasing hours causes the predicted increase.
- Staffing, campaigns and seasonal demand may also matter.

### Average response versus one future month


In [ ]:
new_month <- data.frame(hours = 1200)
knitr::kable(predict(model, newdata = new_month, interval = "confidence"), digits = 1)
knitr::kable(predict(model, newdata = new_month, interval = "prediction"), digits = 1)
range(marketing$hours)
par(mfrow = c(2, 2))
plot(model)
par(mfrow = c(1, 1))


- The confidence interval concerns the **mean response** at 1,200 hours under the fitted model.
- The prediction interval concerns **one future month’s** response and is wider.
- Check the observed hours range before treating a prediction as interpolation.
- Residual plots help assess linearity, spread, unusual observations and normality.
- These plots alone cannot establish independence across months; the file has no month identifier.

### Independent task B

1.  Report the fitted equation with units and interpret its slope.
2.  Compare the two intervals at 1,200 hours and explain their different uses.
3.  Predict at 2,000 hours; explain the additional risk using the observed hours range.
4.  Identify one omitted business factor and describe how to collect it.
5.  Write a 150-word staffing recommendation conditional on the model assumptions.

## C. Optional advanced work: forecast the subscription base

- A **stock** is the total subscription base at a time; a **flow** is the new subscriptions during a period.
- New subscriptions do not equal net growth: cancellations and other changes also matter.
- A straight-line trend is a baseline for extrapolation, not a complete time-series analysis.


In [ ]:
subscriptions <- subscriptions %>% arrange(month)
ggplot(subscriptions, aes(month, subscriptions)) +
  geom_line(colour = petra_ink) + geom_point() +
  labs(x = "Source month number", y = "Home-delivery subscriptions")
trend <- lm(subscriptions ~ month, data = subscriptions)
future <- data.frame(month = 25:28)
knitr::kable(cbind(future, predict(trend, future, interval = "prediction")), digits = 0)
plot(subscriptions$month, residuals(trend), type = "b",
     xlab = "Source month number", ylab = "Trend residual (subscriptions)")
abline(h = 0, col = petra_ink)
acf(residuals(trend), main = "Trend residual autocorrelation")


- `predict()` extrapolates the fitted line to months 25–28.
- Its nominal 95% prediction intervals assume a suitable linear model with independent, constant-variance normal errors.
- Residual patterns or autocorrelation undermine those intervals; they are not validated forecast coverage.
- With only 24 months, seasonal structure is difficult to establish reliably.

### Independent task C

1.  Compare the first and last subscription totals; distinguish absolute growth from percentage growth.
2.  Assess the residual plots and explain whether the linear trend is an adequate baseline.
3.  Write a cautious four-month capacity briefing and explain why a 12-month extrapolation would need more evidence.

## Submission

- Submit the sections assigned by your instructor; the entire case is not required for every assignment.
- For each assigned section: one compact table, one labelled chart and a recommendation with a limitation.
- Keep the three datasets and their units separate.
